# Módulo 01 · Aula 03 — Coleções e Comprehensions

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

Na aula anterior o agrupamento por cidade ficou feio: laços aninhados, varreduras repetidas. Esta aula resolve isso.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | Dicionários | A estrutura de dados mais importante de Python |
| 2 | Métodos de `dict` | `get`, `items`, `setdefault`... |
| 3 | Agrupamento e contagem | O padrão de ouro da análise de dados |
| 4 | `collections` (`defaultdict`, `Counter`) | Atalhos de nível profissional |
| 5 | Conjuntos (`set`) | Unicidade e operações de conjunto |
| 6 | Comprehensions | Transformar coleções em uma linha |
| 7 | Estruturas aninhadas | O formato que a API vai devolver (JSON) |

## 1. Dicionários — pares chave → valor

Um `dict` mapeia **chaves** a **valores**. Se lista é "coisas em ordem", dicionário é "coisas com nome".

```python
d = {"chave": "valor", "outra": 42}
```

**Propriedades:**

- Acesso por chave é **O(1)** — instantâneo, não importa o tamanho. É por isso que ele resolve o agrupamento.
- Chaves devem ser **imutáveis/hasheáveis**: `str`, `int`, `float`, `bool`, `tuple`. Lista **não** pode ser chave.
- Chaves são **únicas** — atribuir de novo sobrescreve.
- Desde o Python 3.7, mantêm a **ordem de inserção**.

In [ ]:
pedido = {
    "id": 1042,
    "cliente": "Maria Souza",
    "cidade": "Campinas",
    "valor": 2599.90,
    "itens": 2,
    "pago": True,
}

print(pedido)
print()
print("Cliente:", pedido["cliente"])
print("Valor:  ", pedido["valor"])
print("Chaves: ", list(pedido.keys()))
print("Tamanho:", len(pedido))

In [ ]:
# Criar, alterar e remover
pedido["frete"] = 9.90            # cria uma chave nova
pedido["valor"] = 2650.00         # sobrescreve
del pedido["itens"]               # remove

print(pedido)
print()
print("Tem 'frete'?  ", "frete" in pedido)
print("Tem 'itens'?  ", "itens" in pedido)

### `[]` vs `.get()` — a diferença que evita crashes

- `d["chave"]` → levanta `KeyError` se a chave não existe.
- `d.get("chave")` → devolve `None` se não existe.
- `d.get("chave", padrao)` → devolve `padrao` se não existe.

Em processamento de dados reais (onde campos faltam o tempo todo), `.get()` com padrão é seu melhor amigo.

In [ ]:
pedido = {"id": 1042, "cidade": "Campinas"}

# print(pedido["cupom"])   # <- descomente: KeyError

print("get sem padrão :", pedido.get("cupom"))
print("get com padrão :", pedido.get("cupom", "SEM_CUPOM"))
print("get existente  :", pedido.get("cidade", "N/D"))

### Percorrendo dicionários

| Método | Devolve | Uso no `for` |
|--------|---------|--------------|
| `.keys()` | as chaves | `for k in d:` (padrão) |
| `.values()` | os valores | `for v in d.values():` |
| `.items()` | pares `(chave, valor)` | `for k, v in d.items():` ← **o mais usado** |

In [ ]:
faturamento = {
    "Campinas": 45300.00,
    "São Paulo": 128900.50,
    "Sorocaba": 22100.00,
    "Ribeirão Preto": 67450.75,
}

for cidade in faturamento:                    # itera nas CHAVES
    print(cidade, end="  ")

print("\n")

for cidade, valor in faturamento.items():     # o idioma canônico
    print(f"{cidade:<18} R$ {valor:>12,.2f}")

print()
print("Total:", f"R$ {sum(faturamento.values()):,.2f}")
print("Maior praça:", max(faturamento, key=faturamento.get))

### Outros métodos úteis

In [ ]:
config = {"host": "localhost", "porta": 5432, "db": "atlas"}

# update — mescla outro dict
config.update({"porta": 5433, "usuario": "admin"})
print(config)

# pop — remove e devolve
usuario = config.pop("usuario")
print("\nremovido:", usuario)

# setdefault — devolve o valor; se a chave não existir, cria com o padrão
timeout = config.setdefault("timeout", 30)
print("timeout:", timeout)
print(config)

# ordenando por valor
faturamento = {"Campinas": 45300.0, "São Paulo": 128900.5, "Sorocaba": 22100.0}
ranking = sorted(faturamento.items(), key=lambda item: item[1], reverse=True)
print("\nRanking:", ranking)

## 2. O padrão de ouro: agrupar e contar com `dict`

Aqui está a razão de existir desta aula. Compare com o bloco feio da aula anterior.

**O padrão:**

```python
acumulador = {}
for registro in dados:
    chave = registro["campo_de_agrupamento"]
    acumulador[chave] = acumulador.get(chave, 0) + registro["valor"]
```

Uma passada só pelos dados. `O(n)` em vez de `O(n²)`.

In [ ]:
pedidos = [
    {"id": 1001, "cidade": "Campinas",       "produto": "Notebook", "qtd": 2,  "preco": 2599.90, "status": "pago"},
    {"id": 1002, "cidade": "São Paulo",      "produto": "Mouse",    "qtd": 10, "preco": 89.90,   "status": "pago"},
    {"id": 1003, "cidade": "Campinas",       "produto": "Teclado",  "qtd": 3,  "preco": 249.00,  "status": "cancelado"},
    {"id": 1004, "cidade": "Sorocaba",       "produto": "Monitor",  "qtd": 1,  "preco": 1199.00, "status": "pago"},
    {"id": 1005, "cidade": "São Paulo",      "produto": "Notebook", "qtd": 1,  "preco": 2599.90, "status": "pago"},
    {"id": 1006, "cidade": "Campinas",       "produto": "Monitor",  "qtd": 4,  "preco": 1199.00, "status": "pago"},
    {"id": 1007, "cidade": "Ribeirão Preto", "produto": "Mouse",    "qtd": 25, "preco": 89.90,   "status": "pago"},
    {"id": 1008, "cidade": "São Paulo",      "produto": "Teclado",  "qtd": 6,  "preco": 249.00,  "status": "pendente"},
    {"id": 1009, "cidade": "Campinas",       "produto": "Notebook", "qtd": 1,  "preco": 2599.90, "status": "pago"},
    {"id": 1010, "cidade": "Sorocaba",       "produto": "Mouse",    "qtd": 8,  "preco": 89.90,   "status": "pago"},
]

# AGRUPAMENTO EM 4 LINHAS
por_cidade = {}
for p in pedidos:
    if p["status"] != "pago":
        continue
    por_cidade[p["cidade"]] = por_cidade.get(p["cidade"], 0.0) + p["qtd"] * p["preco"]

for cidade, total in sorted(por_cidade.items(), key=lambda x: x[1], reverse=True):
    print(f"{cidade:<18} R$ {total:>12,.2f}")

In [ ]:
# CONTAGEM de ocorrências — mesmo padrão, somando 1
contagem_status = {}
for p in pedidos:
    contagem_status[p["status"]] = contagem_status.get(p["status"], 0) + 1

print(contagem_status)

# Contagem de itens vendidos por produto
itens_por_produto = {}
for p in pedidos:
    if p["status"] == "pago":
        itens_por_produto[p["produto"]] = itens_por_produto.get(p["produto"], 0) + p["qtd"]

print()
for produto, itens in sorted(itens_por_produto.items(), key=lambda x: -x[1]):
    barra = "█" * (itens // 2)
    print(f"{produto:<10} {itens:>3} {barra}")

### Agrupar registros inteiros (dicionário de listas)

Às vezes você não quer somar, quer **juntar os registros**. Aí o valor do dicionário é uma lista, e `setdefault` brilha.

In [ ]:
agrupado = {}
for p in pedidos:
    agrupado.setdefault(p["cidade"], []).append(p["id"])

for cidade, ids in agrupado.items():
    print(f"{cidade:<18} {len(ids)} pedidos: {ids}")

## 3. `collections` — os atalhos profissionais

O módulo `collections` da biblioteca padrão traz versões especializadas de `dict`.

- **`defaultdict(tipo)`** — dicionário que cria automaticamente o valor padrão para chaves novas. Elimina o `.get(k, 0)` e o `.setdefault(k, [])`.
- **`Counter(iteravel)`** — conta ocorrências e ainda oferece `.most_common(n)`.

In [ ]:
from collections import defaultdict, Counter

# defaultdict(float) — chave nova nasce valendo 0.0
por_cidade = defaultdict(float)
for p in pedidos:
    if p["status"] == "pago":
        por_cidade[p["cidade"]] += p["qtd"] * p["preco"]

print(dict(por_cidade))

# defaultdict(list) — chave nova nasce como lista vazia
por_produto = defaultdict(list)
for p in pedidos:
    por_produto[p["produto"]].append(p["id"])

print()
for produto, ids in por_produto.items():
    print(f"{produto:<10} {ids}")

In [ ]:
# Counter — contagem em uma linha
status = Counter(p["status"] for p in pedidos)
cidades = Counter(p["cidade"] for p in pedidos)

print("Status: ", status)
print("Top 2 cidades por nº de pedidos:", cidades.most_common(2))

# Counter também conta caracteres, palavras...
texto = "engenharia de dados e engenharia de software"
print("\nPalavras mais comuns:", Counter(texto.split()).most_common(3))

## 4. Conjuntos (`set`)

Um `set` é uma coleção **não ordenada** de elementos **únicos** e **hasheáveis**.

```python
s = {1, 2, 3}
vazio = set()      # ⚠️ {} cria um DICIONÁRIO vazio, não um set!
```

**Para que serve:**

1. **Remover duplicatas** instantaneamente.
2. **Testar pertencimento** em O(1) — muito mais rápido que `in` numa lista grande.
3. **Álgebra de conjuntos**: união, interseção, diferença.

In [ ]:
cidades_julho = {"Campinas", "São Paulo", "Sorocaba", "Campinas", "Jundiaí"}
print(cidades_julho)                 # duplicata sumiu sozinha
print("Tamanho:", len(cidades_julho))

# Removendo duplicatas de uma lista (perde a ordem)
com_repetidos = ["Mouse", "Teclado", "Mouse", "Monitor", "Teclado", "Mouse"]
print("\nÚnicos:", set(com_repetidos))
print("Únicos preservando ordem:", list(dict.fromkeys(com_repetidos)))

In [ ]:
julho = {"Campinas", "São Paulo", "Sorocaba", "Jundiaí"}
agosto = {"Campinas", "São Paulo", "Ribeirão Preto", "Santos"}

print("União        (| ):", julho | agosto)
print("Interseção   (& ):", julho & agosto)
print("Diferença    (- ):", julho - agosto, "  <- só compraram em julho")
print("Dif. simétrica(^):", julho ^ agosto, " <- em um mês ou outro, não nos dois")
print()
print("Novas praças em agosto:", agosto - julho)
print("Praças perdidas:       ", julho - agosto)
print("Fiéis:                 ", julho & agosto)

In [ ]:
# Métodos de set
tags = {"eletrônicos", "informática"}

tags.add("promoção")
tags.update(["frete-grátis", "black-friday"])
tags.discard("inexistente")     # discard não reclama se não existir
tags.remove("promoção")         # remove levanta KeyError se não existir

print(sorted(tags))
print("É subconjunto de todas as tags?", tags <= {"eletrônicos", "informática", "frete-grátis", "black-friday", "extra"})

### Por que `set` é rápido: uma demonstração

In [ ]:
import time

n = 200_000
lista_grande = list(range(n))
set_grande = set(lista_grande)
alvo = n - 1     # pior caso para a lista: o último elemento

t0 = time.perf_counter()
for _ in range(200):
    alvo in lista_grande
t_lista = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(200):
    alvo in set_grande
t_set = time.perf_counter() - t0

print(f"lista: {t_lista*1000:8.2f} ms")
print(f"set:   {t_set*1000:8.4f} ms")
print(f"set é ~{t_lista/max(t_set, 1e-9):,.0f}x mais rápido nesta busca")

## 5. Comprehensions

Sintaxe compacta para **construir** uma coleção a partir de outra. Substitui o padrão "cria vazia → laço → append".

```python
# forma longa
resultado = []
for x in colecao:
    if condicao(x):
        resultado.append(transforma(x))

# comprehension
resultado = [transforma(x) for x in colecao if condicao(x)]
```

Leia sempre da esquerda para a direita: **o que produzir** → **de onde vem** → **filtro**.

⚠️ **Legibilidade acima de esperteza.** Comprehension com dois `for` e três `if` aninhados é pior que o laço explícito. Se não cabe confortavelmente em uma linha mental, use `for`.

In [ ]:
precos = [2599.90, 89.90, 249.00, 1199.00, 45.00, 3199.00]

# List comprehension simples
com_imposto = [round(p * 1.18, 2) for p in precos]
print("Com imposto:", com_imposto)

# Com filtro
caros = [p for p in precos if p > 1000]
print("Acima de 1000:", caros)

# Com transformação e filtro
promocao = [round(p * 0.85, 2) for p in precos if p > 1000]
print("Promoção nos caros:", promocao)

# Com if/else (aqui o ternário vem ANTES do for)
faixas = ["caro" if p > 1000 else "barato" for p in precos]
print("Faixas:", faixas)

In [ ]:
# Comprehension sobre dicionários (o caso real)
faturamento_por_produto = {}
for p in pedidos:
    if p["status"] == "pago":
        faturamento_por_produto[p["produto"]] = faturamento_por_produto.get(p["produto"], 0) + p["qtd"] * p["preco"]

# Extrair só os IDs dos pedidos pagos
ids_pagos = [p["id"] for p in pedidos if p["status"] == "pago"]
print("IDs pagos:", ids_pagos)

# Extrair cidades únicas
cidades_unicas = {p["cidade"] for p in pedidos}          # SET comprehension
print("Cidades:", sorted(cidades_unicas))

# DICT comprehension: id -> valor total
totais = {p["id"]: round(p["qtd"] * p["preco"], 2) for p in pedidos}
print("Totais:", totais)

# Inverter um dicionário
siglas = {"Campinas": "CPS", "São Paulo": "SAO", "Sorocaba": "SOD"}
invertido = {v: k for k, v in siglas.items()}
print("\nInvertido:", invertido)

# Filtrar um dicionário
grandes = {k: v for k, v in faturamento_por_produto.items() if v > 2000}
print("Produtos acima de 2000:", {k: round(v, 2) for k, v in grandes.items()})

In [ ]:
# Comprehension aninhada — achatando uma matriz
matriz = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

achatada = [valor for linha in matriz for valor in linha]
print("Achatada:", achatada)

# Leia como: for linha in matriz: for valor in linha: yield valor
# A ordem dos 'for' é a MESMA do laço aninhado equivalente.

# Construindo uma matriz
tabuada = [[i * j for j in range(1, 6)] for i in range(1, 4)]
for linha in tabuada:
    print(linha)

### Generator expressions: comprehension preguiçosa

Trocando `[]` por `()`, você cria um **gerador**: ele não constrói a lista na memória, produz os itens sob demanda.

Use quando for apenas consumir o resultado uma vez — típico dentro de `sum()`, `max()`, `any()`, `all()`.

In [ ]:
import sys

lista = [x * 2 for x in range(100_000)]
gerador = (x * 2 for x in range(100_000))

print("Memória da lista:   ", f"{sys.getsizeof(lista):>9,} bytes")
print("Memória do gerador: ", f"{sys.getsizeof(gerador):>9,} bytes")

# Uso típico: agregação sem materializar a lista
total_pago = sum(p["qtd"] * p["preco"] for p in pedidos if p["status"] == "pago")
print(f"\nFaturamento pago: R$ {total_pago:,.2f}")
print("Algum pedido acima de 5k?", any(p["qtd"] * p["preco"] > 5000 for p in pedidos))

## 6. Estruturas aninhadas

Dados reais raramente são planos. Um `dict` cujos valores são `list` de `dict` é exatamente a forma de um JSON — o formato que sua API vai devolver no Módulo 06.

In [ ]:
empresa = {
    "nome": "Aurora Comércio",
    "sede": {"cidade": "Campinas", "uf": "SP"},
    "fundacao": 2019,
    "filiais": [
        {"cidade": "São Paulo",      "funcionarios": 42, "ativa": True},
        {"cidade": "Ribeirão Preto", "funcionarios": 12, "ativa": True},
        {"cidade": "Santos",         "funcionarios": 8,  "ativa": False},
    ],
}

print("Sede:", empresa["sede"]["cidade"], "-", empresa["sede"]["uf"])
print("Total de filiais:", len(empresa["filiais"]))
print()

for filial in empresa["filiais"]:
    marca = "🟢" if filial["ativa"] else "🔴"
    print(f"{marca} {filial['cidade']:<18} {filial['funcionarios']:>3} funcionários")

ativos = sum(f["funcionarios"] for f in empresa["filiais"] if f["ativa"])
print(f"\nFuncionários em filiais ativas: {ativos}")

In [ ]:
# Acesso seguro em estruturas aninhadas
dados = {"cliente": {"nome": "Maria", "endereco": {"cidade": "Campinas"}}}

# Encadeando .get com padrão — nunca quebra
uf = dados.get("cliente", {}).get("endereco", {}).get("uf", "N/D")
cidade = dados.get("cliente", {}).get("endereco", {}).get("cidade", "N/D")

print("Cidade:", cidade)
print("UF:    ", uf)

## 🔧 Prática guiada — Dashboard Aurora

Agora com dicionários, o mesmo relatório da aula anterior fica curto, rápido e com muito mais métricas.

In [ ]:
from collections import defaultdict, Counter

pagos = [p for p in pedidos if p["status"] == "pago"]

# --- Agregações em uma passada ---
metricas_cidade = defaultdict(lambda: {"faturamento": 0.0, "pedidos": 0, "itens": 0})

for p in pagos:
    m = metricas_cidade[p["cidade"]]
    m["faturamento"] += p["qtd"] * p["preco"]
    m["pedidos"] += 1
    m["itens"] += p["qtd"]

faturamento_total = sum(m["faturamento"] for m in metricas_cidade.values())
mix_produto = Counter()
for p in pagos:
    mix_produto[p["produto"]] += p["qtd"]

# --- Relatório ---
print("╔" + "═" * 76 + "╗")
print("║" + "ATLAS · DASHBOARD DE VENDAS — AURORA COMÉRCIO".center(76) + "║")
print("╚" + "═" * 76 + "╝")

print(f"\nPedidos recebidos : {len(pedidos)}")
print(f"Pedidos faturados : {len(pagos)}  ({len(pagos)/len(pedidos):.0%})")
print(f"Faturamento total : R$ {faturamento_total:,.2f}")
print(f"Ticket médio      : R$ {faturamento_total/len(pagos):,.2f}")
print(f"Praças ativas     : {len(metricas_cidade)}")

print("\n" + "─" * 78)
print(f"{'CIDADE':<18}{'FATURAMENTO':>16}{'PEDIDOS':>10}{'ITENS':>8}{'TICKET':>12}{'SHARE':>9}")
print("─" * 78)

ordenado = sorted(metricas_cidade.items(), key=lambda kv: kv[1]["faturamento"], reverse=True)
for cidade, m in ordenado:
    ticket = m["faturamento"] / m["pedidos"]
    share = m["faturamento"] / faturamento_total
    print(f"{cidade:<18}{m['faturamento']:>16,.2f}{m['pedidos']:>10}{m['itens']:>8}{ticket:>12,.2f}{share:>9.1%}")

print("─" * 78)
print(f"{'TOTAL':<18}{faturamento_total:>16,.2f}{len(pagos):>10}{sum(m['itens'] for m in metricas_cidade.values()):>8}")

print("\nMIX DE PRODUTOS (unidades vendidas)")
print("─" * 40)
for produto, unidades in mix_produto.most_common():
    barra = "▇" * unidades
    print(f"{produto:<10}{unidades:>4}  {barra}")

# --- Status ---
print("\nSTATUS DOS PEDIDOS")
print("─" * 40)
for status, qtd in Counter(p["status"] for p in pedidos).most_common():
    print(f"{status:<12}{qtd:>3}  ({qtd/len(pedidos):.0%})")

## 📝 Exercícios rápidos

**E1.** A partir de `pedidos`, construa um `dict` que mapeie **produto → faturamento total** (apenas pedidos pagos), usando `defaultdict`.

**E2.** Use uma dict comprehension para criar `{cidade: nº de pedidos}` a partir de `pedidos`.

**E3.** Dadas duas listas de e-mails (`base_2025` e `base_2026`), use `set` para responder: quantos clientes são novos, quantos foram perdidos e quantos permaneceram.

**E4.** Dado `texto = "o rato roeu a roupa do rei de roma"`, use `Counter` para achar as 3 letras mais frequentes (ignorando espaços).

**E5.** Transforme a lista `pedidos` em um dict indexado por `id` (`{1001: {...}, 1002: {...}}`) usando dict comprehension.

**E6.** Escreva uma comprehension que produza a lista de IDs dos pedidos cujo valor total (`qtd * preco`) ultrapassa R$ 2.000.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3
base_2025 = {"ana@x.com", "bruno@x.com", "carla@x.com", "diego@x.com"}
base_2026 = {"bruno@x.com", "carla@x.com", "elisa@x.com", "fabio@x.com"}

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

## ✅ Checklist de saída

- [ ] Sei criar, ler, atualizar e remover chaves de um `dict`
- [ ] Uso `.get(chave, padrao)` para não quebrar com chave ausente
- [ ] Percorro dicionários com `.items()`
- [ ] Domino o padrão de **agrupamento** `acc[k] = acc.get(k, 0) + v`
- [ ] Sei usar `defaultdict` e `Counter`
- [ ] Sei quando um `set` é a estrutura certa (unicidade, pertencimento, álgebra)
- [ ] Escrevo list, dict e set comprehensions com filtro
- [ ] Sei a diferença entre `[...]` e `(...)` (lista vs gerador)
- [ ] Navego em estruturas aninhadas (dict de list de dict)

---

### ➡️ Próxima aula

**`01_04_Funcoes_e_Modulos.ipynb`** — Funções, escopo LEGB, módulos e `if __name__ == "__main__"`. Onde seu código para de ser um bloco só e vira software reutilizável.